In [1]:
import sys
print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
c:\Users\agarw\Desktop\Celebal\Week5_Spark_Questions\.venv-1\Scripts\python.exe


In [3]:
import sys
!{sys.executable} -m pip install pyspark

  Using cached pyspark-4.2.0-py2.py3-none-any.whl
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.2.0


## Q1: Limitations of Traditional MapReduce

Traditional MapReduce has the following limitations:
•	It stores intermediate results on disk, which makes processing slow. 
•	It is inefficient for iterative algorithms such as Machine Learning. 
•	It has higher latency for interactive queries. 
•	It requires multiple MapReduce jobs for complex workflows. 
•	Data processing is more complex because of repeated disk I/O. 
Spark is preferred because it supports in-memory processing, faster iterative computations, interactive queries, and a simpler programming model.


## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark can cache or persist data in memory using operations such as:
df.cache()
In traditional disk-based systems, every iteration reads data from disk and writes intermediate results back to disk. Spark stores frequently used data in RAM, allowing subsequent iterations to access it much faster.
This is especially useful for algorithms such as:
K-Means 
Logistic Regression 
Decision Trees 
PageRank 
Therefore, Spark significantly reduces disk I/O and improves the performance of iterative Machine Learning algorithms.


## Q3: Remove Duplicate Rows

Remove duplicate rows based on `user_id` and `transaction_date`.

In [2]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType
)
from datetime import date

schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("amount", IntegerType(), True)
])

data = [
    ("user1", date(2023, 1, 1), 100),
    ("user2", date(2023, 1, 1), 150),
    ("user1", date(2023, 1, 2), 200),
    ("user2", date(2023, 1, 1), 150)
]

df = spark.createDataFrame(data, schema)

df_unique = df.dropDuplicates(
    ["user_id", "transaction_date"]
)

df_unique.show()

+-------+----------------+------+
|user_id|transaction_date|amount|
+-------+----------------+------+
|  user1|      2023-01-01|   100|
|  user2|      2023-01-01|   150|
|  user1|      2023-01-02|   200|
+-------+----------------+------+



## Q4: Average Sale Amount by Product Category in the West Region

Filter the DataFrame for the West region and calculate the average sale amount for each product category.


In [3]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

sales_schema = StructType([
    StructField("region", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("sale_amount", IntegerType(), True)
])

sales_data = [
    ("West", "Electronics", 1000),
    ("West", "Electronics", 1500),
    ("West", "Furniture", 800),
    ("West", "Furniture", 1200),
    ("East", "Electronics", 2000),
    ("South", "Furniture", 900)
]

df_sales = spark.createDataFrame(
    sales_data,
    sales_schema
)

df_sales.show()

+------+----------------+-----------+
|region|product_category|sale_amount|
+------+----------------+-----------+
|  West|     Electronics|       1000|
|  West|     Electronics|       1500|
|  West|       Furniture|        800|
|  West|       Furniture|       1200|
|  East|     Electronics|       2000|
| South|       Furniture|        900|
+------+----------------+-----------+



In [4]:
from pyspark.sql.functions import col, avg

result = (
    df_sales
    .filter(col("region") == "West")
    .groupBy("product_category")
    .agg(
        avg("sale_amount").alias("average_sale_amount")
    )
)

result.show()

+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|     Electronics|             1250.0|
|       Furniture|             1000.0|
+----------------+-------------------+



## Q5: Handling Null Values

`.na.drop()` removes rows containing null values.

`.na.fill()` replaces null values with a specified value.

In this example, null values in the `status` column are replaced with `"Unknown"`.

In [5]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

status_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("status", StringType(), True)
])

status_data = [
    ("user1", "Active"),
    ("user2", None),
    ("user3", "Inactive")
]

df_status = spark.createDataFrame(
    status_data,
    status_schema
)

df_filled = df_status.na.fill(
    {"status": "Unknown"}
)

df_filled.show()

+-------+--------+
|user_id|  status|
+-------+--------+
|  user1|  Active|
|  user2| Unknown|
|  user3|Inactive|
+-------+--------+



## Q6: Count Records by City

Group the DataFrame by city, count the records, and keep only cities where the count is greater than 100.

In [7]:
from pyspark.sql import Row
from pyspark.sql.functions import count, col

# Create sample data
data = (
    [("Delhi",) for _ in range(120)] +
    [("Mumbai",) for _ in range(150)] +
    [("Haridwar",) for _ in range(80)]
)

df_city = spark.createDataFrame(data, ["city"])

# Count records for each city
result = (
    df_city
    .groupBy("city")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 100)
)

result.show()

+------+------------+
|  city|record_count|
+------+------------+
| Delhi|         120|
|Mumbai|         150|
+------+------------+



## Q7: DataFrame Immutability

Spark DataFrames are immutable, which means that they cannot be changed directly after they are created.

Data cleaning operations such as dropping columns or renaming columns create a new DataFrame. The original DataFrame remains unchanged.

For example:

df_new = df.drop("unwanted_column")

df_renamed = df.withColumnRenamed("old_name", "new_name")

## Q8: Filter Premium Subscribers Aged 18 to 30

Filter rows where the age is between 18 and 30, inclusive, and the subscription is Premium.

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("subscription", StringType(), True)
])

data = [
    ("user1", 20, "Premium"),
    ("user2", 25, "Basic"),
    ("user3", 30, "Premium"),
    ("user4", 17, "Premium"),
    ("user5", 35, "Premium")
]

df_age = spark.createDataFrame(data, schema)

df_age.show()

+-------+---+------------+
|user_id|age|subscription|
+-------+---+------------+
|  user1| 20|     Premium|
|  user2| 25|       Basic|
|  user3| 30|     Premium|
|  user4| 17|     Premium|
|  user5| 35|     Premium|
+-------+---+------------+



In [10]:
from pyspark.sql.functions import col

result = df_age.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

result.show()

+-------+---+------------+
|user_id|age|subscription|
+-------+---+------------+
|  user1| 20|     Premium|
|  user3| 30|     Premium|
+-------+---+------------+



## Q9: Handling Null Values Before Aggregation

Null values should often be handled before mathematical aggregations such as `sum()` and `avg()` because missing values can affect the accuracy and interpretation of the results.

Depending on the situation, null values can be removed or replaced with an appropriate value. For example, missing prices may be replaced with 0 when a null value represents no revenue.

Cleaning null values first makes the aggregation results more reliable and consistent.

## Q10: Cast and Rename Timestamp Column

Cast the `raw_timestamp` column to `TimestampType` and rename it to `event_time`.

In [11]:
from pyspark.sql.types import TimestampType
from pyspark.sql.functions import col

data = [
    ("2026-07-20 10:30:00",),
    ("2026-07-20 12:45:00",)
]

df_timestamp = spark.createDataFrame(
    data,
    ["raw_timestamp"]
)

df_event = (
    df_timestamp
    .withColumn(
        "raw_timestamp",
        col("raw_timestamp").cast(TimestampType())
    )
    .withColumnRenamed(
        "raw_timestamp",
        "event_time"
    )
)

df_event.show()
df_event.printSchema()

+-------------------+
|         event_time|
+-------------------+
|2026-07-20 10:30:00|
|2026-07-20 12:45:00|
+-------------------+

root
 |-- event_time: timestamp (nullable = true)



## Q11: Shuffle Process

A Shuffle occurs when Spark redistributes data between partitions during operations such as `groupBy()`, `join()`, and `distinct()`.

For example, during a groupBy operation, records with the same key may be located in different partitions. Spark moves these records across the network so that all records with the same key are brought together.

It is called a wide transformation because an output partition may depend on data from multiple input partitions.

Shuffle operations can be expensive because they involve network communication, data movement, and sometimes disk I/O.

## Q12: Remove Invalid Email and Username Records

Remove rows where the email is null OR the username is an empty string.

In [12]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col

schema = StructType([
    StructField("email", StringType(), True),
    StructField("username", StringType(), True)
])

data = [
    ("user1@gmail.com", "user1"),
    (None, "user2"),
    ("user3@gmail.com", ""),
    ("user4@gmail.com", "user4")
]

df_users = spark.createDataFrame(data, schema)

df_cleaned = df_users.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

df_cleaned.show()

+---------------+--------+
|          email|username|
+---------------+--------+
|user1@gmail.com|   user1|
|user4@gmail.com|   user4|
+---------------+--------+



## Q13: Multiple Aggregations Using agg()

The agg() function allows multiple aggregate functions to be applied to a column at the same time.

In [13]:
from pyspark.sql.functions import min, max, avg

data = [
    (100,),
    (250,),
    (500,),
    (750,)
]

df_price = spark.createDataFrame(
    data,
    ["price"]
)

result = df_price.agg(
    min("price").alias("minimum_price"),
    max("price").alias("maximum_price"),
    avg("price").alias("average_price")
)

result.show()

+-------------+-------------+-------------+
|minimum_price|maximum_price|average_price|
+-------------+-------------+-------------+
|          100|          750|        400.0|
+-------------+-------------+-------------+



## Q14: Risk of inferSchema=True

Using inferSchema=True with messy or inconsistent date formats can cause Spark to infer an incorrect data type or fail to correctly parse some date values.

For example, dates may appear in different formats such as:

2026-07-20
20/07/2026
July 20, 2026

Spark may not interpret all these values consistently. This can lead to null values, incorrect data types, failed transformations, and incorrect sorting or filtering.

Therefore, it is safer to explicitly define the schema and specify the date format when working with inconsistent date data.

## Q15: Complete Data Processing Pipeline

The following pipeline removes duplicate records, fills null prices with 0, and calculates the total revenue for each store.

In [14]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

from pyspark.sql.functions import sum

schema = StructType([
    StructField("store_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", IntegerType(), True)
])

data = [
    ("S1", "Laptop", 1000),
    ("S1", "Phone", None),
    ("S2", "Tablet", 500),
    ("S1", "Laptop", 1000),
    ("S2", "Monitor", 700)
]

df_store = spark.createDataFrame(data, schema)

result = (
    df_store
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(
        sum("price").alias("total_revenue")
    )
)

result.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|      S2|         1200|
|      S1|         1000|
+--------+-------------+

